# Solution 01: Zero-Shot NER with GLiNER Bi-Encoder

In [1]:
from gliner import GLiNER
import json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "news_articles.json")) as f:
    articles = json.load(f)

print(f"✓ Loaded {len(articles)} articles")

✓ Loaded 10 articles


## Part 1: Load Model and Extract Entities

In [2]:
model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")

text = articles[0]["text"]
labels = ["person", "organization", "location", "date"]
entities = model.predict_entities(text, labels, threshold=0.3)

print(f"Text: {text}\n")
for e in entities:
    print(f"  [{e['label']:12}] {e['text']!r:30} score={e['score']:.2f}")

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/gliner/model.py:447: UserWarning: Resizing embeddings is not supported for bi-encoder models.
  instance.resize_embeddings()


Text: Elon Musk announced that Tesla will open a new Gigafactory in Austin, Texas on March 22, 2024, creating over 5,000 manufacturing jobs and boosting the local economy.

  [person      ] 'Elon Musk'                    score=0.73
  [organization] 'Tesla'                        score=0.81
  [organization] 'Gigafactory'                  score=0.42
  [location    ] 'Austin'                       score=0.49
  [location    ] 'Texas'                        score=0.62
  [date        ] 'March 22, 2024'               score=0.82


In [3]:
# TEST — model loaded
assert 'model' in dir() or 'model' in vars(), "Load the model into variable 'model'"
assert hasattr(model, 'predict_entities'), "model must have .predict_entities() method"
print("✓ model loaded")

✓ model loaded


In [4]:
# TEST — entity format
assert isinstance(entities, list)
assert len(entities) > 0
required_keys = {"text", "label", "start", "end", "score"}
for i, e in enumerate(entities):
    assert not (required_keys - e.keys()), f"Entity {i} missing keys"
for e in entities:
    assert text[e["start"]:e["end"]] == e["text"]
print(f"✓ Found {len(entities)} entities with correct format")

✓ Found 6 entities with correct format


## Part 2: Batch Extraction Function

In [5]:
def extract_entities(model, texts, labels, threshold=0.3):
    """Extract entities from a list of texts using GLiNER batch inference."""
    return model.inference(texts, labels, threshold=threshold)


labels = ["person", "organization", "location", "date"]
texts = [a["text"] for a in articles]
results = extract_entities(model, texts, labels)

assert isinstance(results, list)
assert len(results) == len(articles)
for i, res in enumerate(results):
    assert isinstance(res, list), f"Result {i} must be a list"

total = sum(len(r) for r in results)
print(f"✓ Processed {len(results)} articles, {total} total entities")
for i, (article, count) in enumerate(zip(articles, [len(r) for r in results])):
    persons = [e['text'] for e in results[i] if e['label'] == 'person']
    print(f"  [{article['domain']:15}] {count} entities  persons={persons}")

✓ Processed 10 articles, 58 total entities
  [tech           ] 6 entities  persons=['Elon Musk']
  [sports         ] 7 entities  persons=['Cristiano Ronaldo']
  [tech           ] 6 entities  persons=['Tim Cook']
  [science        ] 6 entities  persons=[]
  [finance        ] 5 entities  persons=['Jamie Dimon']
  [tech           ] 6 entities  persons=['Sam Altman']
  [entertainment  ] 5 entities  persons=['Steven Spielberg']
  [climate        ] 6 entities  persons=['António Guterres']
  [healthcare     ] 4 entities  persons=[]
  [philanthropy   ] 7 entities  persons=['Jeff Bezos', 'Lauren Sánchez']


## Part 3: Pre-Compute Label Embeddings

In [6]:
labels = ["person", "organization", "location", "date"]

# Step 1: encode labels once
entity_embeddings = model.encode_labels(labels, batch_size=8)
print(f"✓ entity_embeddings shape: {entity_embeddings.shape}")

assert entity_embeddings is not None
assert hasattr(entity_embeddings, 'shape')
assert entity_embeddings.shape[0] == len(labels)

Encoding labels:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding labels: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]

Encoding labels: 100%|██████████| 1/1 [00:00<00:00,  1.03it/s]

✓ entity_embeddings shape: torch.Size([4, 384])


In [7]:
texts_sample = [a["text"] for a in articles[:5]]

# Step 2: inference with pre-computed embeddings
results_precomputed = model.batch_predict_with_embeds(
    texts_sample, entity_embeddings, labels, threshold=0.3
)

assert isinstance(results_precomputed, list)
assert len(results_precomputed) == 5

# Compare with direct run
results_direct = model.inference(texts_sample, labels, threshold=0.3)
pre_counts = [len(r) for r in results_precomputed]
dir_counts = [len(r) for r in results_direct]

print(f"✓ Pre-computed: {pre_counts}")
print(f"  Direct:       {dir_counts}")
print("✓ Pre-computed embeddings produce consistent results")

✓ Pre-computed: [6, 7, 6, 6, 5]
  Direct:       [6, 7, 6, 6, 5]
✓ Pre-computed embeddings produce consistent results
